<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W7D1_Daily_Challenge_Book_Text_Analysis_WordCloud_BoW_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semaine 7 — Jour 1  
# Défi quotidien : Analyse de livres avec WordCloud, BoW et TF-IDF

**Developers Institute & Sira Labs**  
**COT GenAI & Machine Learning Bootcamp — 2026**

## Ouvrages étudiés

1. *Alice's Adventures in Wonderland*
2. *Through the Looking-Glass and What Alice Found There*
3. *A Tangled Tale*

## Objectifs

Ce notebook permet de :

- télécharger et nettoyer des textes de Project Gutenberg ;
- supprimer les en-têtes, crédits, licences et autres parties non pertinentes ;
- tokeniser les textes ;
- supprimer les stopwords ;
- appliquer le stemming et la lemmatisation ;
- effectuer le POS tagging ;
- extraire les entités nommées avec NLTK ;
- créer un nuage de mots pour chaque livre ;
- analyser les mots avec Bag of Words ;
- comparer les résultats avec TF-IDF.

> Dans Google Colab, exécutez les cellules dans l'ordre avec  
> **Runtime → Run all**.

## Environnement

Google Colab utilise un environnement Python isolé pour chaque session.  
Pour une exécution locale, il est recommandé de créer et d'activer un environnement virtuel avant d'installer les bibliothèques.

In [ ]:
# Installation des bibliothèques nécessaires

!pip install -q requests nltk spacy wordcloud matplotlib pandas numpy scikit-learn
!python -m spacy download en_core_web_sm -q

In [ ]:
# Importation des bibliothèques et téléchargement des ressources NLTK

import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import nltk
import spacy

from IPython.display import display
from nltk import pos_tag, word_tokenize, sent_tokenize, ne_chunk_sents
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tree import Tree
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

nltk_resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words"
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print("Bibliothèques et ressources prêtes.")

# 1. Chargement et nettoyage des textes

La fonction `load_texts()` :

1. reçoit une liste d'URL ;
2. télécharge chaque texte ;
3. supprime l'en-tête et la licence Project Gutenberg grâce aux marqueurs `START` et `END` ;
4. repère le début réel du livre afin d'écarter les pages de titre et tables des matières ;
5. élimine les caractères qui ne correspondent pas à des mots grâce aux expressions régulières ;
6. retourne le corpus brut pertinent et le corpus nettoyé.

In [ ]:
# URLs et titres

URLS = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "https://www.gutenberg.org/cache/epub/12/pg12.txt",
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt"
]

TITLES = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale"
]

# Motifs permettant de repérer le début narratif réel de chaque livre.
STORY_START_PATTERNS = {
    URLS[0]: r"CHAPTER I\.\s+Down the Rabbit-Hole\s+Alice was beginning",
    URLS[1]: r"CHAPTER I\.\s+Looking-Glass house\s+One thing was certain",
    URLS[2]: r"A TANGLED TALE\.\s+KNOT I\.\s+EXCELSIOR\.\s+[\"“]Goblin"
}


def extract_gutenberg_body(text, url):
    '''
    Supprime l'en-tête et la licence Gutenberg, puis isole le texte utile.
    '''
    start_pattern = re.compile(
        r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        flags=re.IGNORECASE | re.DOTALL
    )
    end_pattern = re.compile(
        r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        flags=re.IGNORECASE | re.DOTALL
    )

    start_match = start_pattern.search(text)
    end_match = end_pattern.search(text)

    start_index = start_match.end() if start_match else 0
    end_index = end_match.start() if end_match else len(text)

    body = text[start_index:end_index]

    # Retrait des pages de titre, crédits internes et tables des matières.
    story_pattern = STORY_START_PATTERNS.get(url)
    if story_pattern:
        story_match = re.search(
            story_pattern,
            body,
            flags=re.IGNORECASE | re.DOTALL
        )
        if story_match:
            body = body[story_match.start():]

    # Suppression des publicités ou notes placées après la fin du livre.
    end_of_story = re.search(r"\bTHE END\b", body)
    if end_of_story:
        body = body[:end_of_story.start()]

    return body.strip()


def clean_non_words(text):
    '''
    Nettoie les caractères non lexicaux avec des expressions régulières.
    Les apostrophes sont normalisées, les chiffres et la ponctuation sont retirés.
    '''
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[^A-Za-z'\s-]", " ", text)
    text = re.sub(r"[_-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_texts(urls):
    '''
    Télécharge les URL et retourne :
    - raw_corpus : textes pertinents avec ponctuation et casse ;
    - cleaned_corpus : textes dont les caractères non lexicaux ont été retirés.
    '''
    raw_corpus = []
    cleaned_corpus = []

    headers = {
        "User-Agent": "Mozilla/5.0 NLP Educational Notebook"
    }

    for url in urls:
        response = requests.get(url, headers=headers, timeout=60)
        response.raise_for_status()

        # Les fichiers Gutenberg fournis sont encodés en UTF-8.
        downloaded_text = response.content.decode("utf-8-sig", errors="replace")

        relevant_text = extract_gutenberg_body(downloaded_text, url)
        cleaned_text = clean_non_words(relevant_text)

        raw_corpus.append(relevant_text)
        cleaned_corpus.append(cleaned_text)

    return raw_corpus, cleaned_corpus


raw_corpus, cleaned_corpus = load_texts(URLS)

print(f"{len(cleaned_corpus)} livres ont été chargés avec succès.")

In [ ]:
# 2. Affichage des 200 premiers caractères de chaque texte

for title, raw_text, cleaned_text in zip(TITLES, raw_corpus, cleaned_corpus):
    print("=" * 100)
    print(title.upper())
    print("-" * 100)

    print("Premiers caractères du texte pertinent :")
    print(re.sub(r"\s+", " ", raw_text[:200]))

    print("\nPremiers caractères après nettoyage regex :")
    print(cleaned_text[:200])
    print()

### Analyse des parties non pertinentes

Les fichiers téléchargés ne contiennent pas uniquement le récit. Ils comprennent également :

- la présentation du projet Gutenberg ;
- les informations bibliographiques ;
- les crédits de transcription ;
- les conditions d'utilisation et la licence ;
- parfois une page de titre, une table des matières ou des publicités placées après le récit.

Ces parties peuvent modifier artificiellement les fréquences des mots. La fonction précédente les retire en utilisant les marqueurs Gutenberg, le début narratif propre à chaque œuvre et l'expression `THE END`.

# 2. Tokenisation

Les textes sont convertis en minuscules, puis tokenisés.  
Nous conservons uniquement les tokens alphabétiques.

In [ ]:
# 3. Tokenisation et affichage des 150 premiers tokens

tokenized_corpus = []

for title, text in zip(TITLES, cleaned_corpus):
    tokens = [
        token.lower()
        for token in word_tokenize(text)
        if token.isalpha()
    ]

    tokenized_corpus.append(tokens)

    print("=" * 100)
    print(title.upper())
    print(f"Nombre total de tokens : {len(tokens):,}")
    print("\n150 premiers tokens :")
    print(tokens[:150])
    print()

# 3. Suppression des stopwords

Les stopwords sont des mots très fréquents comme `the`, `and`, `i` ou `we`.  
Ils sont souvent peu discriminants pour une analyse de fréquence.

In [ ]:
# 4. Suppression des stopwords avec NLTK

english_stopwords = set(stopwords.words("english"))

tokens_without_stopwords = [
    [token for token in tokens if token not in english_stopwords]
    for tokens in tokenized_corpus
]

for title, original_tokens, filtered_tokens in zip(
    TITLES,
    tokenized_corpus,
    tokens_without_stopwords
):
    print("=" * 100)
    print(title.upper())
    print(f"Tokens avant suppression : {len(original_tokens):,}")
    print(f"Tokens après suppression : {len(filtered_tokens):,}")
    print("50 premiers tokens sans stopwords :")
    print(filtered_tokens[:50])
    print()

In [ ]:
# Vérification avec count() sur plusieurs stopwords

stopwords_to_check = [
    "i", "me", "my", "myself",
    "we", "our", "ours", "ourselves"
]

verification_rows = []

for document_number, (title, before, after) in enumerate(
    zip(TITLES, tokenized_corpus, tokens_without_stopwords),
    start=1
):
    for stopword in stopwords_to_check:
        verification_rows.append({
            "Document": document_number,
            "Livre": title,
            "Stopword": stopword,
            "Avant": before.count(stopword),
            "Après": after.count(stopword)
        })

stopword_verification_df = pd.DataFrame(verification_rows)
display(stopword_verification_df)

assert stopword_verification_df["Après"].sum() == 0
print("Vérification réussie : les stopwords recherchés ont été supprimés.")

# 4. Stemming avec PorterStemmer

Le stemming réduit les mots à une racine approximative.  
Cette racine n'est pas toujours un mot anglais valide.

In [ ]:
# 5. Stemming et affichage des 50 premiers tokens

stemmer = PorterStemmer()

stemmed_corpus = [
    [stemmer.stem(token) for token in tokens]
    for tokens in tokens_without_stopwords
]

for title, stemmed_tokens in zip(TITLES, stemmed_corpus):
    print("=" * 100)
    print(title.upper())
    print("50 premiers tokens stemmés :")
    print(stemmed_tokens[:50])
    print()

# 5. Lemmatisation avec spaCy

La lemmatisation utilise les informations linguistiques du mot pour produire sa forme canonique.  
Par exemple, `children` peut devenir `child` et `was` peut devenir `be`.

In [ ]:
# 6. Lemmatisation avec en_core_web_sm

# Le parser et la NER spaCy ne sont pas nécessaires ici.
nlp_lemma = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

texts_without_stopwords = [
    " ".join(tokens)
    for tokens in tokens_without_stopwords
]

nlp_lemma.max_length = max(len(text) for text in texts_without_stopwords) + 10_000

lemmatized_corpus = []

for doc in nlp_lemma.pipe(texts_without_stopwords, batch_size=1):
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and token.lemma_.strip()
    ]
    lemmatized_corpus.append(lemmas)

for title, lemmas in zip(TITLES, lemmatized_corpus):
    print("=" * 100)
    print(title.upper())
    print("50 premiers tokens lemmatisés :")
    print(lemmas[:50])
    print()

In [ ]:
# Comparaison directe entre stemming et lemmatisation

comparison_rows = []

for title, original, stems, lemmas in zip(
    TITLES,
    tokens_without_stopwords,
    stemmed_corpus,
    lemmatized_corpus
):
    limit = min(30, len(original), len(stems), len(lemmas))

    for index in range(limit):
        comparison_rows.append({
            "Livre": title,
            "Token original": original[index],
            "Stem": stems[index],
            "Lemma": lemmas[index]
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

### 7. Différence entre stemming et lemmatisation

Le **stemming** applique surtout des règles de découpage. Il peut produire des formes incomplètes qui n'existent pas réellement, par exemple `curious` peut être réduit à une racine simplifiée.

La **lemmatisation** utilise le vocabulaire et la catégorie grammaticale pour retrouver une forme canonique correcte. Elle est donc généralement plus précise, mais aussi plus lente.

Dans cette analyse :

- les stems sont utiles pour regrouper rapidement des variantes ;
- les lemmes sont plus faciles à lire et plus adaptés aux nuages de mots, au BoW et à TF-IDF ;
- les différences apparaissent surtout pour les verbes irréguliers, les pluriels et les mots possédant des suffixes.

# 6. POS tagging avec NLTK

Le POS tagging associe une catégorie grammaticale à chaque mot :

- `NN` : nom singulier ;
- `NNS` : nom pluriel ;
- `JJ` : adjectif ;
- `VB` : verbe ;
- `VBD` : verbe au passé ;
- `RB` : adverbe ;
- `NNP` : nom propre.

In [ ]:
# 8. Identification des POS tags de chaque texte

# Nous conservons la casse pour améliorer la reconnaissance des noms propres.
case_sensitive_tokens = [
    [
        token
        for token in word_tokenize(text)
        if token.isalpha()
    ]
    for text in cleaned_corpus
]

pos_tagged_corpus = [
    pos_tag(tokens)
    for tokens in case_sensitive_tokens
]

for title, tagged_tokens in zip(TITLES, pos_tagged_corpus):
    tag_counts = Counter(tag for _, tag in tagged_tokens)

    print("=" * 100)
    print(title.upper())
    print("100 premiers couples (mot, POS) :")
    print(tagged_tokens[:100])

    print("\n15 POS tags les plus fréquents :")
    display(
        pd.DataFrame(
            tag_counts.most_common(15),
            columns=["POS tag", "Fréquence"]
        )
    )

# 7. Extraction des entités nommées avec NLTK

La fonction suivante analyse toutes les phrases par lots afin de limiter l'utilisation de la mémoire.

> Cette cellule est la plus coûteuse du notebook.  
> Par défaut, elle traite toutes les phrases. Pour un test rapide, remplacez temporairement `MAX_NER_SENTENCES = None` par une valeur comme `300`.

In [ ]:
# 9. Extraction des entités de chaque texte avec NLTK

def extract_named_entities_nltk(text, max_sentences=None, batch_size=200):
    '''
    Extrait les entités nommées d'un texte avec NLTK.

    Retour :
        liste de tuples (entité, type)
    '''
    sentences = sent_tokenize(text)

    if max_sentences is not None:
        sentences = sentences[:max_sentences]

    entities = []

    for start in range(0, len(sentences), batch_size):
        sentence_batch = sentences[start:start + batch_size]

        tagged_sentences = [
            pos_tag(word_tokenize(sentence))
            for sentence in sentence_batch
        ]

        chunked_sentences = ne_chunk_sents(
            tagged_sentences,
            binary=False
        )

        for tree in chunked_sentences:
            for node in tree:
                if isinstance(node, Tree):
                    entity_text = " ".join(
                        word for word, _ in node.leaves()
                    )
                    entities.append((entity_text, node.label()))

    return entities


MAX_NER_SENTENCES = None

nltk_entities_corpus = []

for title, text in zip(TITLES, raw_corpus):
    print(f"Analyse NER en cours : {title}")

    entities = extract_named_entities_nltk(
        text,
        max_sentences=MAX_NER_SENTENCES
    )

    nltk_entities_corpus.append(entities)

    entity_counts = Counter(entities)

    print(f"Nombre total d'occurrences d'entités : {len(entities):,}")
    print(f"Nombre d'entités distinctes : {len(entity_counts):,}")

    display(
        pd.DataFrame(
            [
                {
                    "Entité": entity,
                    "Type": label,
                    "Fréquence": frequency
                }
                for (entity, label), frequency
                in entity_counts.most_common(30)
            ]
        )
    )

# 8. Nuages de mots

Les textes lemmatisés sans stopwords sont utilisés, car ils :

- réduisent les variantes d'un même mot ;
- conservent des mots lisibles ;
- retirent une grande partie des mots grammaticaux peu informatifs.

In [ ]:
# 1. Création d'un nuage de mots pour chaque livre

lemmatized_documents = [
    " ".join(lemmas)
    for lemmas in lemmatized_corpus
]

for title, document in zip(TITLES, lemmatized_documents):
    wordcloud = WordCloud(
        width=1400,
        height=700,
        background_color="white",
        collocations=False,
        max_words=150
    ).generate(document)

    plt.figure(figsize=(16, 8))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Nuage de mots — {title}")
    plt.tight_layout()
    plt.show()

# 9. Bag of Words

Le Bag of Words compte les occurrences des mots sans tenir compte de leur ordre.

Le corpus lemmatisé sans stopwords est le plus adapté ici :

- le texte brut contient trop de mots fonctionnels ;
- le texte stemmé est moins lisible ;
- le texte lemmatisé regroupe les variantes tout en conservant des mots compréhensibles.

In [ ]:
# 2. Construction du Bag of Words

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(lemmatized_documents)
bow_features = bow_vectorizer.get_feature_names_out()

print("Dimensions de la matrice BoW :", bow_matrix.shape)
print(
    "Interprétation :",
    bow_matrix.shape[0],
    "documents et",
    bow_matrix.shape[1],
    "mots distincts."
)

# Fréquences globales dans les trois livres
global_word_counts = np.asarray(
    bow_matrix.sum(axis=0)
).ravel()

top_5_indices = global_word_counts.argsort()[-5:][::-1]

top_5_bow_df = pd.DataFrame({
    "Mot": bow_features[top_5_indices],
    "Fréquence": global_word_counts[top_5_indices].astype(int)
})

print("\nLes cinq mots les plus fréquents dans l'ensemble du corpus :")
display(top_5_bow_df)

In [ ]:
# 3. Affichage du BoW et interprétation des nombres

print("Représentation sparse de la matrice BoW :")
print(bow_matrix)

# Extraction des coordonnées non nulles
rows, columns = bow_matrix.nonzero()

bow_triplets = []

for document_index, word_index in zip(rows, columns):
    count = int(bow_matrix[document_index, word_index])

    bow_triplets.append({
        "Numéro du document": int(document_index + 1),
        "Index du mot": int(word_index),
        "Mot": bow_features[word_index],
        "Nombre d'occurrences": count
    })

bow_triplets_df = pd.DataFrame(bow_triplets)

print("\nPremières valeurs non nulles du Bag of Words :")
display(bow_triplets_df.head(100))

### Interprétation des nombres du BoW

Dans la représentation du Bag of Words :

- le premier nombre est le **numéro du document** ; Python commence à l'indice 0, mais le tableau précédent l'affiche à partir de 1 ;
- le deuxième nombre est l'**index du mot** dans le vocabulaire créé par `CountVectorizer` ;
- la valeur correspond au **nombre de fois où le mot apparaît dans le document**.

Exemple conceptuel :

`document 1, index 250, fréquence 40`

signifie que le mot situé à l'index 250 apparaît 40 fois dans le premier livre.

In [ ]:
# 4. Graphique à secteurs des cinq mots les plus fréquents

bow_labels = [
    f"{word} ({frequency})"
    for word, frequency in zip(
        top_5_bow_df["Mot"],
        top_5_bow_df["Fréquence"]
    )
]

plt.figure(figsize=(9, 9))
plt.pie(
    top_5_bow_df["Fréquence"],
    labels=bow_labels,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Les 5 mots les plus fréquents dans les trois livres — BoW")
plt.tight_layout()
plt.show()

### 5. Analyse des résultats du BoW

Les mots les plus fréquents peuvent appartenir à deux groupes :

- des mots liés directement aux histoires et aux personnages, comme `alice`, `queen`, `king` ou d'autres noms propres ;
- des mots fréquents dans la narration et les dialogues, comme `say`, `said`, `would`, `one` ou `little`.

Les noms de personnages sont informatifs, car ils révèlent le sujet principal des livres. En revanche, certains verbes de parole ou termes narratifs sont attendus dans un roman et apportent moins d'information.

Le BoW mesure uniquement la fréquence. Il peut donc donner beaucoup d'importance à un mot répété dans tous les livres, même si ce mot ne permet pas de distinguer un ouvrage des autres.

# 10. TF-IDF

TF-IDF augmente le poids des mots importants dans un document mais moins fréquents dans les autres documents.

Conformément à la consigne :

- `min_df=1` conserve les termes présents dans au moins un document ;
- `max_df=2` retire les termes présents dans les trois documents.

In [ ]:
# 1. Construction du corpus TF-IDF

tfidf_vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=2
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    lemmatized_documents
)

tfidf_features = tfidf_vectorizer.get_feature_names_out()

print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)

top_tfidf_by_document = {}

for document_index, title in enumerate(TITLES):
    scores = tfidf_matrix[document_index].toarray().ravel()

    nonzero_indices = np.flatnonzero(scores)
    sorted_indices = nonzero_indices[
        np.argsort(scores[nonzero_indices])[::-1]
    ]

    top_indices = sorted_indices[:5]

    result_df = pd.DataFrame({
        "Mot": tfidf_features[top_indices],
        "Score TF-IDF": scores[top_indices]
    })

    top_tfidf_by_document[title] = result_df

    print("=" * 100)
    print(title.upper())
    display(result_df)

In [ ]:
# 2. Diagrammes circulaires TF-IDF pour chaque document

for title, result_df in top_tfidf_by_document.items():
    labels = [
        f"{word} ({score:.3f})"
        for word, score in zip(
            result_df["Mot"],
            result_df["Score TF-IDF"]
        )
    ]

    plt.figure(figsize=(9, 9))
    plt.pie(
        result_df["Score TF-IDF"],
        labels=labels,
        autopct="%1.1f%%",
        startangle=90
    )
    plt.title(f"Les 5 termes les plus pertinents selon TF-IDF\n{title}")
    plt.tight_layout()
    plt.show()

### Analyse des résultats TF-IDF

Les résultats TF-IDF sont généralement plus distinctifs que ceux du BoW.

- Un mot très fréquent dans les trois livres peut être absent, car `max_df=2` l'exclut.
- Les termes ayant un score élevé sont davantage associés à un livre particulier.
- Les noms de personnages, lieux, objets ou concepts propres à une histoire remontent plus facilement.
- TF-IDF ne comprend pas le sens des mots, mais il corrige une partie du biais créé par la fréquence brute.

Ainsi, le BoW répond surtout à la question **« Quels mots sont les plus répétés ? »**, tandis que TF-IDF répond plutôt à **« Quels mots caractérisent le mieux chaque document par rapport aux autres ? »**.

# Conclusion générale

Cette analyse montre l'importance du prétraitement dans un projet NLP.

1. Les en-têtes et licences Gutenberg doivent être retirés pour ne pas fausser les résultats.
2. La suppression des stopwords concentre l'analyse sur les termes porteurs de sens.
3. Le stemming est rapide, mais peut produire des racines difficiles à interpréter.
4. La lemmatisation conserve des formes plus correctes et lisibles.
5. Le POS tagging décrit la structure grammaticale des textes.
6. La NER repère les personnes, lieux et organisations.
7. Le nuage de mots donne une vue visuelle des thèmes dominants.
8. Le BoW mesure les fréquences brutes.
9. TF-IDF met davantage en évidence les termes spécifiques à chaque livre.

Pour ce corpus, les documents lemmatisés et débarrassés des stopwords constituent une bonne base pour les nuages de mots, le BoW et TF-IDF.